# LeetCode 2255 — Count Prefixes of a Given String bằng Python

Link đề: https://leetcode.com/problems/count-prefixes-of-a-given-string/ · Độ khó: **Easy** · Chủ đề: Array, String

Skeleton LeetCode:

```python
class Solution:
    def countPrefixes(self, words: List[str], s: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Phân biệt **"`w` là tiền tố của `s`"** (bài này) với **"`s` bắt đầu bằng `pref`"** (bài 2185): hai bài giống nhau nhưng **đảo vai trò**.
2. Dùng `s.startswith(w)`, lát cắt `s[:len(w)] == w` và so sánh từng ký tự để kiểm tra tiền tố.
3. Sinh **mọi tiền tố của `s`** (`s[:1], s[:2], ...`) và tận dụng `set` / `Counter` để đếm nhanh các từ trùng nhau.
4. Làm quen **Trie**: đi dọc theo `s` và cộng số từ **kết thúc** tại mỗi nút.
5. So sánh 5 cách, trace, test ngẫu nhiên và tránh lỗi: nhầm chiều `startswith`, quên đếm từ trùng, quên từ dài hơn `s`.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` và một chuỗi `s`, cả hai chỉ gồm chữ cái thường. Hãy trả về **số chuỗi trong `words` là tiền tố của `s`**.

Tiền tố của `s` là một chuỗi con **liên tiếp xuất hiện ở đầu** `s`. Ví dụ với `s = "abc"` các tiền tố là `"a"`, `"ab"`, `"abc"`.

| Ví dụ | words | s | Output |
|---|---|---|---|
| 1 | `["a","b","c","ab","bc","abc"]` | `"abc"` | `3` |
| 2 | `["a","a"]` | `"aa"` | `2` |

- Ví dụ 1: `"a"`, `"ab"`, `"abc"` là tiền tố của `"abc"` → 3. (`"b"`, `"c"`, `"bc"` không phải tiền tố.)
- Ví dụ 2: hai chuỗi `"a"` giống nhau nhưng **đếm riêng** → 2.

Constraints: `1 <= len(words) <= 1000`, `1 <= len(words[i]), len(s) <= 10`, chỉ gồm chữ cái thường tiếng Anh.

In [1]:
examples = [
    (["a", "b", "c", "ab", "bc", "abc"], "abc", 3),
    (["a", "a"], "aa", 2),
]
for words, s, expected in examples:
    good = [w for w in words if s.startswith(w)]
    print("s =", s, "| tiền tố trong words:", good, "| mong đợi:", expected)

s = abc | tiền tố trong words: ['a', 'ab', 'abc'] | mong đợi: 3
s = aa | tiền tố trong words: ['a', 'a'] | mong đợi: 2


## 3. Kiến thức nền

### 3.1. Tiền tố, hậu tố, chuỗi con

Với `s = "abcde"`:

- **tiền tố**: `"a"`, `"ab"`, `"abc"`, ... (bắt đầu ở đầu `s`);
- **hậu tố**: `"e"`, `"de"`, ... (kết thúc ở cuối `s`);
- **chuỗi con**: `"bcd"`, `"cd"`... (liên tiếp ở bất kỳ đâu).

Mọi tiền tố của `s` có dạng `s[:i]` với `1 <= i <= len(s)`.

In [2]:
s = "abcde"
print("tiền tố :", [s[:i] for i in range(1, len(s) + 1)])
print("hậu tố  :", [s[i:] for i in range(len(s))])
print("'bcd' trong s?", "bcd" in s, "| bắt đầu bằng 'bcd'?", s.startswith("bcd"))

tiền tố : ['a', 'ab', 'abc', 'abcd', 'abcde']
hậu tố  : ['abcde', 'bcde', 'cde', 'de', 'e']
'bcd' trong s? True | bắt đầu bằng 'bcd'? False


### 3.2. `str.startswith` — chú ý **chiều** của hàm

`s.startswith(w)` hỏi: "chuỗi `s` có bắt đầu bằng `w` không?" Tức **`w` là tiền tố của `s`**. Đây đúng là điều đề bài cần (vai trò đảo so với bài 2185, nơi ta gọi `w.startswith(pref)`).

In [3]:
s = "abc"
print(s.startswith("ab"), s.startswith("abc"), s.startswith("abcd"), s.startswith("b"))
print("ab".startswith("abc"))        # chiều ngược: 'ab' có bắt đầu bằng 'abc' không? -> False

True True False False
False


### 3.3. Lát cắt và so sánh

`s[:k]` lấy `k` ký tự đầu của `s`. Nếu `w` có độ dài `k` thì "`w` là tiền tố của `s`" ⇔ `len(w) <= len(s)` và `s[:len(w)] == w`. Lát cắt tự co nếu `k > len(s)`, nên khi `w` dài hơn `s`, `s[:len(w)]` ngắn hơn `w` và phép `==` cho `False`.

In [4]:
s = "abc"
for w in ["a", "abc", "abcd", "bc"]:
    print(f"{w!r:6} len={len(w)} s[:len(w)]={s[:len(w)]!r:6} bằng? {s[:len(w)] == w}")

'a'    len=1 s[:len(w)]='a'    bằng? True
'abc'  len=3 s[:len(w)]='abc'  bằng? True
'abcd' len=4 s[:len(w)]='abc'  bằng? False
'bc'   len=2 s[:len(w)]='ab'   bằng? False


### 3.4. `set` và biểu thức tập hợp (set comprehension)

`{s[:i] for i in range(1, len(s) + 1)}` tạo **tập các tiền tố** của `s`. Kiểm tra `w in prefixes` chạy `O(1)` trung bình, nhanh hơn việc so sánh từng từ với `s` nhiều lần.

In [5]:
s = "abc"
prefixes = {s[:i] for i in range(1, len(s) + 1)}
print(prefixes)
print("ab" in prefixes, "bc" in prefixes)

{'a', 'abc', 'ab'}
True False


### 3.5. `collections.Counter` — đếm số lần mỗi từ

`Counter(words)` cho biết mỗi từ xuất hiện bao lần. Nhờ vậy nếu `"a"` xuất hiện 5 lần và `"a"` là tiền tố của `s`, ta cộng thẳng `5` thay vì kiểm tra từng bản.

In [6]:
from collections import Counter
words = ["a", "b", "a", "ab", "a"]
cnt = Counter(words)
print(cnt, cnt["a"], cnt["zzz"])
print(sum(cnt[p] for p in ["a", "ab", "abc"]))

Counter({'a': 3, 'b': 1, 'ab': 1}) 3 0
4


### 3.6. `dict` lồng nhau cho Trie

Trie là cây tiền tố: mỗi nút là một `dict` từ ký tự sang nút con. Ta thêm một khóa đặc biệt (ví dụ `"#"`) để đếm số từ **kết thúc** tại nút đó.

In [7]:
trie = {}
for w in ["a", "ab", "a"]:
    node = trie
    for ch in w:
        node = node.setdefault(ch, {})
    node["#"] = node.get("#", 0) + 1
print(trie)

{'a': {'#': 2, 'b': {'#': 1}}}


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1: `s = "abc"`. Liệt kê mọi tiền tố của `s`: `"a"`, `"ab"`, `"abc"`. Với mỗi từ trong `words` hỏi: "từ này có nằm trong danh sách tiền tố không?"

| từ | là tiền tố của `abc`? |
|---|---|
| `a` | **có** |
| `b` | không (không bắt đầu ở đầu) |
| `c` | không |
| `ab` | **có** |
| `bc` | không |
| `abc` | **có** |

Đếm được 3. Hai hướng nghĩ:

- **hướng từ `words`**: với mỗi từ, kiểm tra `s.startswith(w)`;
- **hướng từ `s`**: chỉ có `len(s) ≤ 10` tiền tố; đếm số từ bằng đúng mỗi tiền tố rồi cộng lại.

In [8]:
words, s = ["a", "b", "c", "ab", "bc", "abc"], "abc"
print([w for w in words if s.startswith(w)])
prefixes = [s[:i] for i in range(1, len(s) + 1)]
print(prefixes, [words.count(p) for p in prefixes], sum(words.count(p) for p in prefixes))

['a', 'ab', 'abc']
['a', 'ab', 'abc'] [1, 1, 1] 3


## 5. Cách 1 — Brute Force / baseline: so từng ký tự

Với mỗi từ `w`: nếu `w` dài hơn `s` thì loại; ngược lại so `w[i]` với `s[i]` cho `i` từ `0` đến `len(w) - 1`. Chỉ cần một ký tự lệch là loại.

In [9]:
def is_prefix_manual(w, s):
    if len(w) > len(s):
        return False
    for i in range(len(w)):
        if w[i] != s[i]:
            return False
    return True

print([is_prefix_manual(w, "abc") for w in ["a", "ab", "abc", "abcd", "b"]])

[True, True, True, False, False]


In [10]:
from typing import List

class SolutionBrute:
    def countPrefixes(self, words: List[str], s: str) -> int:
        count = 0
        for w in words:
            if len(w) > len(s):
                continue
            ok = True
            for i in range(len(w)):
                if w[i] != s[i]:
                    ok = False
                    break
            if ok:
                count += 1
        return count

print(SolutionBrute().countPrefixes(["a", "b", "c", "ab", "bc", "abc"], "abc"))
print(SolutionBrute().countPrefixes(["a", "a"], "aa"))

3
2


**Độ phức tạp:** Gọi `n = len(words)`, `m = len(s)`. Mỗi từ so tối đa `m` ký tự → **Time `O(n·m)`**, **Space `O(1)`**.

**Nút thắt:** vòng lặp tự viết dài và dễ sai chỉ số. Python có `startswith` cho đúng việc này.

## 6. Ý tưởng thuật toán / bước đột phá

Insight: `w` là tiền tố của `s` ⇔ `s.startswith(w)`. Quy tắc:

- nếu `s.startswith(w)` thì cộng 1; từ dài hơn `s` tự động `False`;
- vì `len(s) ≤ 10`, `s` chỉ có **tối đa 10 tiền tố**. Ta có thể đảo hướng: dựng tập tiền tố của `s` một lần, rồi chỉ cần tra `w in prefixes` — hoặc đếm bằng `Counter(words)` cho từng tiền tố.

Chỉ cần xét `m` giá trị tiền tố thay vì `n` phép so chuỗi dài. Khi `n` lớn và `m` nhỏ, đó là cải thiện đáng kể.

## 7. Các cách tiếp theo

### Cách 2 — `startswith` + `sum`

Dòng ngắn nhất, đúng định nghĩa.

In [11]:
from typing import List

class SolutionStartsWith:
    def countPrefixes(self, words: List[str], s: str) -> int:
        return sum(s.startswith(w) for w in words)

print(SolutionStartsWith().countPrefixes(["a", "b", "c", "ab", "bc", "abc"], "abc"))
print(SolutionStartsWith().countPrefixes(["a", "a"], "aa"))

3
2


**Độ phức tạp:** Time `O(n·m)`, Space `O(1)`.

### Cách 3 — Cắt lát `s[:len(w)] == w`

Giống ý tưởng cách 2 nhưng tường minh hơn: lấy phần đầu của `s` có cùng độ dài với `w` rồi so sánh.

In [12]:
from typing import List

class SolutionSlice:
    def countPrefixes(self, words: List[str], s: str) -> int:
        return sum(1 for w in words if len(w) <= len(s) and s[:len(w)] == w)

print(SolutionSlice().countPrefixes(["a", "b", "c", "ab", "bc", "abc"], "abc"))
print(SolutionSlice().countPrefixes(["a", "a"], "aa"))
print(SolutionSlice().countPrefixes(["abcd"], "abc"))

3
2
0


**Độ phức tạp:** Time `O(n·m)` (mỗi lát cắt tạo chuỗi `≤ m`), Space `O(m)`. Kiểm tra `len(w) <= len(s)` giúp loại sớm từ quá dài.

### Cách 4 — Tập tiền tố của `s` (hash set)

Dựng `prefixes = {s[:i] ...}` một lần (tối đa `m` phần tử), rồi đếm các từ thuộc tập đó. Mỗi phép tra `w in prefixes` tốn `O(|w|)` để băm nhưng **không phụ thuộc `m`** theo kiểu so từng ký tự với `s`; với `|w| ≤ m` thì vẫn `O(m)` nhưng hằng số nhỏ và code rõ ràng.

In [13]:
from typing import List

class SolutionSet:
    def countPrefixes(self, words: List[str], s: str) -> int:
        prefixes = {s[:i] for i in range(1, len(s) + 1)}
        return sum(w in prefixes for w in words)

print(SolutionSet().countPrefixes(["a", "b", "c", "ab", "bc", "abc"], "abc"))
print(SolutionSet().countPrefixes(["a", "a"], "aa"))

3
2


**Độ phức tạp:** Time `O(m²)` dựng tập (mỗi lát cắt dài tới `m`) + `O(n·m)` tra; Space `O(m²)`. Với `m ≤ 10` là hằng số nhỏ.

### Cách 5 — `Counter(words)` rồi cộng theo từng tiền tố của `s`

Đếm trước số lần mỗi từ. Sau đó chỉ duyệt **`m` tiền tố** của `s` và cộng `cnt[prefix]`. Số phép lặp chính chỉ là `m`, bất kể `n` lớn bao nhiêu (sau khi đã đếm).

In [14]:
from collections import Counter
from typing import List

class SolutionCounter:
    def countPrefixes(self, words: List[str], s: str) -> int:
        cnt = Counter(words)
        return sum(cnt[s[:i]] for i in range(1, len(s) + 1))

print(SolutionCounter().countPrefixes(["a", "b", "c", "ab", "bc", "abc"], "abc"))
print(SolutionCounter().countPrefixes(["a", "a"], "aa"))

3
2


**Độ phức tạp:** Time `O(n·L + m²)` với `L` là độ dài từ trung bình (đếm `Counter`) và `m` bước cộng; Space `O(n·L)`. Điểm hay: khi **có nhiều chuỗi `s` truy vấn** trên cùng `words`, `Counter` dựng một lần dùng lại nhiều lần.

### Cách 6 (mở rộng) — Trie và đi dọc theo `s`

Chèn các từ vào Trie, mỗi nút có `end` = số từ kết thúc tại nút. Sau đó đi theo `s`: tại mỗi ký tự đi xuống nút con (nếu không có thì dừng), và cộng `end` của nút vừa đến. Tổng cộng là số từ là tiền tố của `s`.

In [15]:
from typing import List

class TrieNode:
    def __init__(self):
        self.children = {}
        self.end = 0

class SolutionTrie:
    def countPrefixes(self, words: List[str], s: str) -> int:
        root = TrieNode()
        for w in words:
            node = root
            for ch in w:
                if ch not in node.children:
                    node.children[ch] = TrieNode()
                node = node.children[ch]
            node.end += 1
        node = root
        total = 0
        for ch in s:
            if ch not in node.children:
                break
            node = node.children[ch]
            total += node.end
        return total

print(SolutionTrie().countPrefixes(["a", "b", "c", "ab", "bc", "abc"], "abc"))
print(SolutionTrie().countPrefixes(["a", "a"], "aa"))

3
2


**Độ phức tạp:** xây cây `O(tổng độ dài các từ)`, đi dọc `s` `O(m)`; Space `O(tổng độ dài)`. Lợi thế rõ khi **nhiều chuỗi `s`** cùng truy vấn.

## 8. TRACE từng bước

Trace Cách 6 (Trie) với `words = ["a","b","c","ab","bc","abc"]`, `s = "abc"`:

| bước | ký tự `ch` | đi xuống nút | `end` của nút | `total` |
|---|---|---|---|---|
| 1 | `a` | `a` | 1 (từ `"a"`) | 1 |
| 2 | `b` | `ab` | 1 (từ `"ab"`) | 2 |
| 3 | `c` | `abc` | 1 (từ `"abc"`) | 3 |

Kết quả `3`. Các từ `"b"`, `"c"`, `"bc"` nằm ở **nhánh khác** của gốc nên không bao giờ được cộng.

In [16]:
def trace_trie(words, s):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["#"] = node.get("#", 0) + 1
    print("các nhánh ở gốc:", sorted(k for k in root))
    node, total, path = root, 0, ""
    for step, ch in enumerate(s, 1):
        if ch not in node:
            print(f"bước {step}: ký tự {ch!r} không có nhánh -> dừng")
            break
        node = node[ch]
        path += ch
        total += node.get("#", 0)
        print(f"bước {step}: ch={ch!r}, nút={path!r}, end={node.get('#', 0)}, total={total}")
    return total

print("kết quả:", trace_trie(["a", "b", "c", "ab", "bc", "abc"], "abc"))
print("kết quả:", trace_trie(["a", "a"], "aa"))

các nhánh ở gốc: ['a', 'b', 'c']
bước 1: ch='a', nút='a', end=1, total=1
bước 2: ch='b', nút='ab', end=1, total=2
bước 3: ch='c', nút='abc', end=1, total=3
kết quả: 3
các nhánh ở gốc: ['a']
bước 1: ch='a', nút='a', end=2, total=2
bước 2: ký tự 'a' không có nhánh -> dừng
kết quả: 2


In [17]:
def trace_startswith(words, s):
    count = 0
    for step, w in enumerate(words, 1):
        ok = s.startswith(w)
        if ok:
            count += 1
        print(f"bước {step}: w={w!r:6} s.startswith(w)={ok!s:5} count={count}")
    return count

print("kết quả:", trace_startswith(["a", "b", "c", "ab", "bc", "abc"], "abc"))

bước 1: w='a'    s.startswith(w)=True  count=1
bước 2: w='b'    s.startswith(w)=False count=1
bước 3: w='c'    s.startswith(w)=False count=1
bước 4: w='ab'   s.startswith(w)=True  count=2
bước 5: w='bc'   s.startswith(w)=False count=2
bước 6: w='abc'  s.startswith(w)=True  count=3
kết quả: 3


## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def countPrefixes(self, words: List[str], s: str) -> int:
        return sum(s.startswith(w) for w in words)

print(Solution().countPrefixes(["a", "b", "c", "ab", "bc", "abc"], "abc"))
print(Solution().countPrefixes(["a", "a"], "aa"))

3
2


## 10. Vì sao thuật toán đúng

- **Định nghĩa:** `w` là tiền tố của `s` ⇔ `len(w) ≤ len(s)` và `s[i] == w[i]` với mọi `i < len(w)` ⇔ `s.startswith(w)`. Cách 2 và 3 kiểm tra đúng điều kiện này cho từng phần tử của `words`.
- **Đếm đúng từ trùng:** ta đếm theo **phần tử mảng** (mỗi lần xuất hiện cộng 1), nên hai từ `"a"` cho 2.
- **Cách tập tiền tố / Counter:** mọi tiền tố của `s` là `s[:i]`, `1 ≤ i ≤ m`; một từ là tiền tố của `s` ⇔ nó bằng đúng một trong `m` chuỗi đó. Các tiền tố **đôi một khác nhau** (độ dài khác nhau) nên mỗi từ khớp với tối đa một tiền tố — cộng `cnt[prefix]` không đếm trùng.
- **Trie:** đường đi theo `s` qua các nút ứng với `s[:1], s[:2], ...`. `end` của nút `s[:i]` là số từ bằng đúng `s[:i]`. Tổng các `end` trên đường đi chính là số từ là tiền tố.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(words)`, `m = len(s)` (`≤ 10`), `T` = tổng độ dài các từ.

| Cách | Time | Space |
|---|---|---|
| 1. So từng ký tự | `O(n·m)` | `O(1)` |
| 2. `startswith` | `O(n·m)` | `O(1)` |
| 3. Cắt lát | `O(n·m)` | `O(m)` |
| 4. Tập tiền tố | `O(m² + n·m)` | `O(m²)` |
| 5. Counter theo tiền tố | `O(T + m²)` | `O(T)` |
| 6. Trie | `O(T + m)` | `O(T)` |

Vì `m ≤ 10` và `n ≤ 1000`, cả sáu cách đều chạy tức thời; sự khác biệt chỉ quan trọng khi mở rộng sang nhiều truy vấn `s`.

## 12. Bộ test

Ví dụ đề, edge case (từ dài hơn `s`, từ đúng bằng `s`, từ trùng nhiều lần, không có tiền tố nào) và test ngẫu nhiên đối chiếu **tất cả cách**.

In [19]:
import random

solvers = [SolutionBrute(), SolutionStartsWith(), SolutionSlice(), SolutionSet(), SolutionCounter(), SolutionTrie(), Solution()]

fixed = [
    (["a", "b", "c", "ab", "bc", "abc"], "abc", 3),
    (["a", "a"], "aa", 2),
    (["abc"], "abc", 1),                      # từ đúng bằng s
    (["abcd"], "abc", 0),                     # từ dài hơn s
    (["b", "c", "bc"], "abc", 0),             # phản ví dụ: là chuỗi con nhưng không phải tiền tố
    (["z"] * 5 + ["a"], "azz", 1),            # nhiều từ trùng không khớp
    (["a"] * 1000, "a", 1000),                # biên số lượng
    (["x", "xy", "xyz", "xyzz"], "xyz", 3),
]
for words, s, expected in fixed:
    for sol in solvers:
        assert sol.countPrefixes(list(words), s) == expected, (type(sol).__name__, words, s)
print("test cố định: OK")

def reference(words, s):
    return sum(1 for w in words if len(w) <= len(s) and all(w[i] == s[i] for i in range(len(w))))

random.seed(2255)
for _ in range(500):
    n = random.randint(1, 15)
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 8)))
    words = ["".join(random.choice("ab") for _ in range(random.randint(1, 9))) for _ in range(n)]
    expected = reference(words, s)
    for sol in solvers:
        assert sol.countPrefixes(list(words), s) == expected, (type(sol).__name__, words, s)
print("500 test ngẫu nhiên: OK")

test cố định: OK
500 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Gọi `startswith` ngược chiều

`w.startswith(s)` hỏi "`w` bắt đầu bằng `s`" — nghĩa là `s` là tiền tố của `w`, **ngược với đề**.

In [20]:
words, s = ["a", "ab", "abcd"], "abc"
wrong = sum(w.startswith(s) for w in words)
right = sum(s.startswith(w) for w in words)
print("sai :", wrong, "(chỉ 'abcd' bắt đầu bằng 'abc')")
print("đúng:", right, "('a' và 'ab' là tiền tố của 'abc')")

sai : 1 (chỉ 'abcd' bắt đầu bằng 'abc')
đúng: 2 ('a' và 'ab' là tiền tố của 'abc')


### Lỗi 2: Dùng `in` thay cho tiền tố

`w in s` chỉ kiểm tra `w` là chuỗi con ở **bất kỳ đâu**.

In [21]:
words, s = ["a", "b", "c", "ab", "bc", "abc"], "abc"
print("sai :", sum(w in s for w in words), "| đúng:", sum(s.startswith(w) for w in words))

sai : 6 | đúng: 3


### Lỗi 3: Loại trùng bằng `set(words)`

Đề đếm theo **phần tử** của mảng, không phải theo giá trị khác nhau.

In [22]:
words, s = ["a", "a"], "aa"
print("sai :", sum(s.startswith(w) for w in set(words)), "| đúng:", sum(s.startswith(w) for w in words))

sai : 1 | đúng: 2


### Lỗi 4: Truy cập `s[i]` khi từ dài hơn `s`

Khi tự viết vòng so sánh mà quên kiểm tra độ dài, `s[i]` có thể gây `IndexError`.

In [23]:
def wrong4(w, s):
    for i in range(len(w)):
        if w[i] != s[i]:
            return False
    return True

try:
    wrong4("abcd", "abc")
except IndexError as e:
    print("sai -> IndexError:", e)
print("đúng:", "abc".startswith("abcd"))

sai -> IndexError: string index out of range
đúng: False


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đọc kỹ **ai là tiền tố của ai**: "các từ trong `words` là tiền tố của `s`" khác "các từ trong `words` có tiền tố `pref`". Viết ra: `s.startswith(w)` hay `w.startswith(pref)`?
- `len(s) ≤ 10` rất nhỏ → có thể liệt kê tiền tố của `s` hoặc dùng `startswith` thẳng.
- Dữ liệu có từ trùng → dùng `Counter` hoặc cứ đếm theo vòng lặp, **đừng** loại trùng.
- Nhiều truy vấn trên cùng tập từ → Trie / Counter dựng trước.
- `sum(điều_kiện for x in ds)` là cách đếm gọn trong Python.

## 15. Biến thể của bài

1. LeetCode 2185: đếm số từ **bắt đầu bằng** `pref` (đảo vai trò).
2. LeetCode 3042/3043: đếm cặp `(i, j)` mà `words[i]` vừa là tiền tố vừa là hậu tố của `words[j]`.
3. Đếm các từ là **hậu tố** của `s` (dùng `endswith`).
4. Tìm **tiền tố chung dài nhất** của mọi từ (LeetCode 14 Longest Common Prefix).
5. Cho `q` chuỗi `s` truy vấn: dùng Trie dựng một lần, mỗi truy vấn `O(m)`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng ký tự | vòng lặp + `break` | `O(n·m)` | `O(1)` | học nền tảng |
| 2. `startswith` | `s.startswith(w)` | `O(n·m)` | `O(1)` | **khuyến nghị khi nộp** |
| 3. Cắt lát | `s[:len(w)] == w` | `O(n·m)` | `O(m)` | dễ hiểu, không cần học `startswith` |
| 4. Tập tiền tố | tra `w in prefixes` | `O(m² + n·m)` | `O(m²)` | `m` nhỏ, `n` lớn |
| 5. Counter | cộng `cnt[s[:i]]` | `O(T + m²)` | `O(T)` | nhiều từ trùng, nhiều truy vấn |
| 6. Trie | đi dọc `s`, cộng `end` | `O(T + m)` | `O(T)` | nhiều truy vấn `s` trên cùng `words` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print("hello".startswith("he"), "he".startswith("hello"))` in ra gì?

**Bài 2 (tính tay).** `words = ["c","co","cod","code","codes","o"]`, `s = "code"`. Có bao nhiêu từ là tiền tố của `s`?

**Bài 3 (điền TODO).** Hoàn thành hàm đếm số từ là **hậu tố** của `s`.

In [24]:
def count_suffixes(words, s):
    # TODO: số từ w trong words sao cho s.endswith(w)
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `prefix_lengths(words, s)` trả về **danh sách độ dài** của các từ là tiền tố của `s`, theo thứ tự trong `words`.

In [25]:
def prefix_lengths(words, s):
    # TODO: tự viết
    pass

**Bài 5 (mini challenge).** Cho `words` và danh sách `queries` các chuỗi `s`. Trả về danh sách đáp án (số từ là tiền tố) cho từng truy vấn, **dựng cấu trúc một lần** rồi trả lời nhanh (gợi ý: `Counter` của `words`).

In [26]:
def prefix_counts_for_queries(words, queries):
    # TODO: dựng Counter(words) một lần, mỗi truy vấn cộng cnt[q[:i]]
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"hello"` có bắt đầu bằng `"he"` (True); `"he"` không bắt đầu bằng `"hello"` (False). Bài 2: liệt kê các tiền tố của `"code"`: `c`, `co`, `cod`, `code`.
- **Mức 2 (vừa):** Bài 3 dùng `s.endswith(w)`. Bài 4: list comprehension `[len(w) for w in words if s.startswith(w)]`.
- **Mức 3 (sâu):** Bài 5: `cnt = Counter(words)`; mỗi truy vấn `q`: `sum(cnt[q[:i]] for i in range(1, len(q) + 1))`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
from collections import Counter

# Bài 1 và 2
assert ("hello".startswith("he"), "he".startswith("hello")) == (True, False)
assert Solution().countPrefixes(["c", "co", "cod", "code", "codes", "o"], "code") == 4

# Bài 3
def count_suffixes(words, s):
    return sum(s.endswith(w) for w in words)

# Bài 4
def prefix_lengths(words, s):
    return [len(w) for w in words if s.startswith(w)]

# Bài 5
def prefix_counts_for_queries(words, queries):
    cnt = Counter(words)
    return [sum(cnt[q[:i]] for i in range(1, len(q) + 1)) for q in queries]

assert count_suffixes(["c", "de", "cde", "x", "e"], "abcde") == 3   # "de", "cde", "e"
assert prefix_lengths(["ab", "b", "abc", "a", "abcd"], "abc") == [2, 3, 1]
words = ["a", "ab", "ab", "abc", "b"]
queries = ["abc", "a", "bz", "z"]
assert prefix_counts_for_queries(words, queries) == [Solution().countPrefixes(words, q) for q in queries]
print(prefix_counts_for_queries(words, queries))
print("tất cả bài tập OK")

[4, 1, 1, 0]
tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (từ nào là tiền tố của `s`)

```python
class Solution:
    def countPrefixes(self, words: List[str], s: str) -> int:
        return sum(s.startswith(w) for w in words)
```

**Checklist**

- [ ] Đã viết đúng **chiều**: `s.startswith(w)` (không phải `w.startswith(s)`)?
- [ ] Từ dài hơn `s` được xử lý (không `IndexError`, kết quả `False`)?
- [ ] Từ trùng nhau được đếm đủ (không dùng `set(words)`)?
- [ ] Đã thử phản ví dụ: `"b"`, `"bc"` là chuỗi con nhưng không phải tiền tố?
- [ ] Đã thử từ đúng bằng `s` và danh sách không có tiền tố nào?
- [ ] Signature đúng `countPrefixes(self, words: List[str], s: str) -> int` và có `from typing import List`?